# 🈯 Manga OCR — Detection & Recognition Pipeline



End-to-end demo: detects speech-bubble regions in manga pages and reads

the Japanese text inside each bubble using three back-ends.



| Stage | Tool |

|---|---|

| Bubble detection | EasyOCR |

| Text recognition | SVTR · CRNN (PaddleOCR) · TrOCR (HuggingFace) |

| Interface | Gradio |

## 📦 1 · Install Dependencies

In [1]:
import subprocess, sys

# ── Upgrade build tools (reduces install failures) ───────────────────
print("Upgrading pip ...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade", "pip", "setuptools", "wheel"]
)

# ── PaddlePaddle GPU 3.0.0 (CUDA 12.6) ──────────────────────────────
# Installed alongside PyTorch — no conflict; both frameworks share
# the CUDA driver but operate independently. Do NOT remove PyTorch
# first: EasyOCR depends on it.
print("Installing PaddlePaddle GPU ...")
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "paddlepaddle-gpu==3.0.0",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
    ]
)

# ── Remaining dependencies ────────────────────────────────────────────
print("Installing remaining dependencies ...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q",
     "easyocr", "gradio", "transformers",
     "fugashi", "ipadic", "unidic-lite", "nest_asyncio"]
)

# ── Verify environment ────────────────────────────────────────────────
import paddle, torch
print("\n=== Environment ===")
print(f"  Paddle  {paddle.__version__}  |  CUDA: {paddle.device.is_compiled_with_cuda()}  |  GPUs: {paddle.device.cuda.device_count()}")
print(f"  PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
for i in range(paddle.device.cuda.device_count()):
    print(f"  GPU {i}: {paddle.device.cuda.get_device_name(i)}")

Upgrading pip ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.3 MB/s eta 0:00:00
Installing PaddlePaddle GPU ...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires nvidia-cublas-cu12==12.8.4.1; platform_system == "Linux", but you have nvidia-cublas-cu12 12.6.4.1 which is incompatible.
torch 2.10.0+cu128 requires nvidia-cuda-cupti-cu12==12.8.90; platform_system == "Linux", but you have nvidia-cuda-cupti-cu12 12.6.80 which is incompatible.
torch 2.10.0+cu128 requires nvidia-cuda-nvrtc-cu12==12.8.93; platform_system == "Linux", but you have nvidia-cuda-nvrtc-cu12 12.6.77 which is incompatible.
torch 2.10.0+cu128 requires nvidia-cuda-runtime-cu12==12.8.90; platform_system == "Linux", but you have nvidia-cuda-runtime-cu12 12.6.77 which is incompatible.
torch 2.10.0+cu128 requires nvidia-cudnn-cu12==9.10.2.21; platform_system == "Linux", but you have nvidia-cudnn-cu12 9.5.1.17 which is incompatible.
torch 2.10.0+cu128 requires nvidia-cufft-cu12==11.3.3.

Installing remaining dependencies ...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlepaddle-gpu 3.0.0 requires nvidia-cublas-cu12==12.6.4.1; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
paddlepaddle-gpu 3.0.0 requires nvidia-cuda-cupti-cu12==12.6.80; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.8.90 which is incompatible.
paddlepaddle-gpu 3.0.0 requires nvidia-cuda-nvrtc-cu12==12.6.77; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.8.93 which is incompatible.
paddlepaddle-gpu 3.0.0 requires nvidia-cuda-runtime-cu12==12.6.77; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-runtime-cu12 12.8.90 which is incompatible.
paddlepaddle-gpu 3.0.0 requires nvidia-cudnn-cu12==9.5.


=== Environment ===
  Paddle  3.0.0  |  CUDA: True  |  GPUs: 2
  PyTorch 2.10.0+cu128  |  CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 🛠️ 2 · Imports

In [2]:
import os, re, sys, time, subprocess, yaml
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import cv2
import numpy as np
import torch
import gradio as gr
import easyocr
import nest_asyncio
from PIL import Image
from IPython.display import display
from transformers import AutoImageProcessor, AutoTokenizer, VisionEncoderDecoderModel

nest_asyncio.apply()

## ⚙️ 3 · Configuration



Set all model paths and the `MODEL_MAP` registry here.  

Change only this cell when adapting the notebook to a new environment.

In [3]:
BASE_PATH      = Path("/kaggle/input/models/thoandanh/maga109s-ocr/pytorch/default/3")
TROCR_DIR      = Path("/kaggle/input/models/thoandanh/trocr-for-rec/transformers/default/3"
                      "/trocr-manga109-jpbert-lora-stage2-final-merged")
DICT_PATH      = BASE_PATH / "dict_japanese.txt"
OUTPUT_ROOT    = Path("/kaggle/working")
PADDLEOCR_DIR  = OUTPUT_ROOT / "PaddleOCR"
TEMP_DIR       = OUTPUT_ROOT / "temp_bubbles"
SVTR_CFG_PATH  = OUTPUT_ROOT / "config_SVTR.yml"
CRNN_CFG_PATH  = OUTPUT_ROOT / "config_CRNN.yml"

for d in (OUTPUT_ROOT, TEMP_DIR, OUTPUT_ROOT / 'output' / 'inference'):
    d.mkdir(parents=True, exist_ok=True)

MODEL_MAP = {
    "SVTR": {
        "cfg":   str(SVTR_CFG_PATH),
        "ckpt":  BASE_PATH / "SVTR-manga/best_accuracy/best_accuracy",
        "shape": "3, 48, 320",
    },
    "CRNN": {
        "cfg":   str(CRNN_CFG_PATH),
        "ckpt":  BASE_PATH / "CRNN-manga/best_accuracy/best_accuracy",
        "shape": "3, 32, 320",
    },
    "TrOCR": {"type": "huggingface"},
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config ready  |  device = {DEVICE}")

✅ Config ready  |  device = cuda


## 🏗️ 4 · Bootstrap PaddleOCR & Write Configs

In [26]:
# ── Clone repo ──────────────────────────────────────────────────────
if not PADDLEOCR_DIR.is_dir():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/PaddlePaddle/PaddleOCR.git",
                           str(PADDLEOCR_DIR)])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', str(PADDLEOCR_DIR / 'requirements.txt')])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'scikit-image', 'imgaug', 'lmdb', 'rapidfuzz',
                       'pyclipper', 'shapely', 'opencv-contrib-python'])

# ── SVTR config ──────────────────────────────────────────────────────
svtr_cfg = {
    'Global': {
        'use_gpu': True, 'character_dict_path': str(DICT_PATH),
        'max_text_length': 80, 'use_space_char': False, 'infer_mode': False,
    },
    'Architecture': {
        'model_type': 'rec', 'algorithm': 'SVTR',
        'Transform': {
            'name': 'STN_ON', 'tps_inputsize': [32, 64], 'tps_outputsize': [32, 320],
            'num_control_points': 20, 'tps_margins': [0.05, 0.05], 'stn_activation': 'none',
        },
        'Backbone': {
            'name': 'SVTRNet', 'img_size': [32, 320], 'out_char_num': 40,
            'out_channels': 192, 'patch_merging': 'Conv',
            'embed_dim': [64, 128, 256], 'depth': [3, 6, 3], 'num_heads': [2, 4, 8],
            'mixer': ['Local'] * 6 + ['Global'] * 6,
            'local_mixer': [[7, 11], [7, 11], [7, 11]],
            'last_stage': True, 'prenorm': False,
        },
        'Neck': {'name': 'SequenceEncoder', 'encoder_type': 'reshape'},
        'Head': {'name': 'CTCHead'},
    },
    'Loss': {'name': 'CTCLoss'},
    'PostProcess': {'name': 'CTCLabelDecode'},
    'Metric': {'name': 'RecMetric', 'main_indicator': 'norm_edit_dis'},
    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet', 'data_dir': './',
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'CTCLabelEncode': None},
                {'RecResizeImg': {'image_shape': [3, 32, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label', 'length']}},
            ],
        },
        'loader': {'shuffle': False, 'drop_last': False,
                   'batch_size_per_card': 32, 'num_workers': 4},
    },
}

# ── CRNN config ──────────────────────────────────────────────────────
crnn_cfg = {
    'Global': {
        'use_gpu': True, 'character_dict_path': str(DICT_PATH),
        'max_text_length': 80, 'use_space_char': False, 'infer_mode': False,
    },
    'Architecture': {
        'model_type': 'rec', 'algorithm': 'CRNN', 'Transform': None,
        'Backbone': {
            'name': 'MobileNetV3', 'scale': 0.5,
            'model_name': 'small', 'small_stride': [1, 2, 2, 2],
        },
        'Neck': {'name': 'SequenceEncoder', 'encoder_type': 'rnn', 'hidden_size': 48},
        'Head': {'name': 'CTCHead', 'fc_decay': 1.0e-05},
    },
    'Loss': {'name': 'CTCLoss'},
    'PostProcess': {'name': 'CTCLabelDecode'},
    'Metric': {'name': 'RecMetric', 'main_indicator': 'norm_edit_dis'},
    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet', 'data_dir': './',
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'CTCLabelEncode': None},
                {'RecResizeImg': {'image_shape': [3, 32, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label', 'length']}},
            ],
        },
    },
}

SVTR_CFG_PATH.write_text(yaml.dump(svtr_cfg, allow_unicode=True, sort_keys=False))
CRNN_CFG_PATH.write_text(yaml.dump(crnn_cfg, allow_unicode=True, sort_keys=False))
print("✅ PaddleOCR bootstrapped")
print(f"   SVTR config -> {SVTR_CFG_PATH}")
print(f"   CRNN config -> {CRNN_CFG_PATH}")

✅ PaddleOCR bootstrapped
   SVTR config -> /kaggle/working/config_SVTR.yml
   CRNN config -> /kaggle/working/config_CRNN.yml


## 🫧 5 · Bubble Processing Utilities



Splits each detected bubble crop into individual text columns (or rows for

horizontal layouts), rotates vertical columns upright, then stitches them

into one horizontal strip ready for the OCR model.

In [5]:
@dataclass
class BubbleLineConfig:
    spacer_px:          int   = 10
    min_component_area: int   = 5
    gap_percentile:     float = 15.0
    gap_scale:          float = 1.0
    min_gap_width:      int   = 2
    min_col_width:      int   = 3
    sort_right_to_left: bool  = True
    horizontal_ratio:   float = 1.3
    furigana_ratio:     float = 0.6
    min_row_height:     int   = 5
    iou_threshold:      float = 0.4


def _binarize(bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 25, 15)


def _remove_small_components(mask: np.ndarray, min_area: int) -> np.ndarray:
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = np.zeros_like(mask)
    for i in range(1, n):
        if int(stats[i, cv2.CC_STAT_AREA]) >= min_area:
            out[labels == i] = 255
    return out


def _gap_runs(gap_mask: np.ndarray, min_width: int) -> List[Tuple[int, int]]:
    runs, start = [], None
    for i, is_gap in enumerate(gap_mask):
        if is_gap and start is None:
            start = i
        elif not is_gap and start is not None:
            if i - start >= min_width:
                runs.append((start, i))
            start = None
    if start is not None and len(gap_mask) - start >= min_width:
        runs.append((start, len(gap_mask)))
    return runs


def _projection_intervals(
    profile: np.ndarray, axis_len: int,
    cfg: BubbleLineConfig, min_span: int,
) -> List[Tuple[int, int]]:
    active = profile[profile > 0]
    if active.size == 0:
        return [(0, axis_len)]
    th   = float(np.percentile(active, cfg.gap_percentile) * cfg.gap_scale)
    gaps = _gap_runs(profile <= th, cfg.min_gap_width)
    cuts = sorted({0, axis_len} | {(a + b) // 2 for a, b in gaps})
    spans = [(cuts[i], cuts[i+1]) for i in range(len(cuts)-1)
             if cuts[i+1] - cuts[i] >= min_span]
    return spans or [(0, axis_len)]


def _h_intervals(mask, cfg):
    return _projection_intervals(mask.sum(axis=1).astype(np.float32),
                                 mask.shape[0], cfg, cfg.min_row_height)

def _v_intervals(mask, cfg):
    return _projection_intervals(mask.sum(axis=0).astype(np.float32),
                                 mask.shape[1], cfg, cfg.min_col_width)


def _filter_furigana(intervals, cfg):
    if len(intervals) <= 1:
        return intervals
    th   = max(b - a for a, b in intervals) * cfg.furigana_ratio
    kept = [iv for iv in intervals if iv[1] - iv[0] >= th]
    return kept or intervals


def _fg_x(mask, y0, y1):
    xs = np.where(mask[y0:y1].sum(axis=0) > 0)[0]
    return (int(xs.min()), int(xs.max()) + 1) if xs.size else None

def _fg_y(mask, x0, x1):
    ys = np.where(mask[:, x0:x1].sum(axis=1) > 0)[0]
    return (int(ys.min()), int(ys.max()) + 1) if ys.size else None

def _resize_h(img, h):
    oh, ow = img.shape[:2]
    if oh == h or oh == 0:
        return img
    return cv2.resize(img, (max(1, int(ow * h / oh)), h),
                      interpolation=cv2.INTER_LANCZOS4)


def bubble_to_line(bgr: np.ndarray, cfg: BubbleLineConfig) -> np.ndarray:
    """Convert a bubble crop to a horizontal text strip."""
    h, w  = bgr.shape[:2]
    clean = _remove_small_components(_binarize(bgr), cfg.min_component_area)
    pieces = []

    if w >= h * cfg.horizontal_ratio:
        for y0, y1 in sorted(_filter_furigana(_h_intervals(clean, cfg), cfg)):
            rng = _fg_x(clean, y0, y1)
            if rng:
                pieces.append(bgr[y0:y1, rng[0]:rng[1]])
    else:
        for x0, x1 in sorted(_filter_furigana(_v_intervals(clean, cfg), cfg),
                              reverse=cfg.sort_right_to_left):
            rng = _fg_y(clean, x0, x1)
            if rng:
                pieces.append(cv2.rotate(bgr[rng[0]:rng[1], x0:x1],
                                         cv2.ROTATE_90_COUNTERCLOCKWISE))
    if not pieces:
        return bgr

    ref_h   = pieces[0].shape[0]
    spacer  = np.full((ref_h, cfg.spacer_px, 3), 255, np.uint8)
    strip   = pieces[0]
    for p in pieces[1:]:
        strip = np.concatenate([strip, spacer, _resize_h(p, ref_h)], axis=1)
    return strip


def _iou_filter(boxA, boxB, threshold):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter  = max(0, xB - xA) * max(0, yB - yA)
    if inter <= 0:
        return False
    return inter / ((boxB[2]-boxB[0]) * (boxB[3]-boxB[1])) > 0.8


def filter_overlapping_boxes(results, iou_threshold=0.4):
    if not results:
        return []
    boxes = []
    for res in results:
        pts = np.array(res[0])
        x0, y0 = pts.min(axis=0)
        x1, y1 = pts.max(axis=0)
        boxes.append([x0, y0, x1, y1, res])
    boxes.sort(key=lambda b: (b[2]-b[0])*(b[3]-b[1]), reverse=True)
    kept = []
    while boxes:
        best = boxes.pop(0)
        kept.append(best[4])
        boxes = [b for b in boxes if not _iou_filter(best, b, iou_threshold)]
    return kept


_reader = easyocr.Reader(['vi', 'en'])
_cfg    = BubbleLineConfig()
print("✅ Bubble processing utilities ready")

✅ Bubble processing utilities ready


## 🧠 6 · Load TrOCR Model

In [6]:
print("Loading TrOCR ...")
try:
    _processor = AutoImageProcessor.from_pretrained(TROCR_DIR, local_files_only=True)
    _tokenizer = AutoTokenizer.from_pretrained(TROCR_DIR, local_files_only=True)
    _trocr     = VisionEncoderDecoderModel.from_pretrained(TROCR_DIR, local_files_only=True)

    for cfg_obj in (_trocr.config, _trocr.generation_config):
        cfg_obj.decoder_start_token_id = _tokenizer.cls_token_id
        cfg_obj.bos_token_id           = _tokenizer.cls_token_id
        cfg_obj.pad_token_id           = _tokenizer.pad_token_id
        cfg_obj.eos_token_id           = _tokenizer.sep_token_id
    _trocr.generation_config.max_new_tokens = 96

    _trocr.eval().to(DEVICE)
    print(f"✅ TrOCR loaded on {DEVICE}")
except Exception as err:
    print(f"❌ TrOCR load failed: {err}")

Loading TrOCR ...


Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.bert.embeddings.word_embeddings.weight to decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie decoder.cls.predictions.bias to decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ TrOCR loaded on cuda


## 🔍 7 · Inference Functions

In [35]:
def _normalize(text: str) -> str:
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", text))

def _parse_paddle_log_mapped(raw: str, paths: list[str]) -> list[list[str]]:
    """
    PaddleOCR log format (2 dòng liên tiếp):
      [timestamp] ppocr INFO: infer_img: /path/bubble_001.png
      [timestamp] ppocr INFO: \t result: テキスト\t0.9907
    """
    stem_to_pred: dict[str, str] = {}

    lines = raw.splitlines()
    for i, line in enumerate(lines):
        # Bắt dòng chứa tên file
        m_img = re.search(r"infer_img:\s*(.+\.png)", line)
        if not m_img:
            continue

        filename = Path(m_img.group(1).strip()).name  # "bubble_010.png"

        # Dòng ngay sau phải là result
        if i + 1 >= len(lines):
            continue

        m_res = re.search(r"result:\s*(.+?)\t([0-9.]+)", lines[i + 1])
        if m_res:
            text  = m_res.group(1).strip()
            score = float(m_res.group(2))
            stem_to_pred[filename] = f"{text} ({score:.2f})"

    # Reorder theo thứ tự paths gốc — đảm bảo crop[i] ↔ pred[i]
    results = []
    for path in paths:
        filename = Path(path).name
        pred = stem_to_pred.get(filename)
        results.append([pred] if pred else ["[no result]"])
    return results


def run_paddle_inference(paths: list[str], model_key: str):
    """Run SVTR or CRNN on a batch — results ordered to match input paths."""
    m          = MODEL_MAP[model_key]
    infer_dir  = str(Path(paths[0]).parent)

    cmd = [
        sys.executable, str(PADDLEOCR_DIR / "tools/infer_rec.py"),
        "-c", m["cfg"], "-o",
        f"Global.pretrained_model={m['ckpt']}",
        f"Global.infer_img={infer_dir}",
        f"Global.character_dict_path={DICT_PATH}",
        "Global.use_space_char=False",
        f"Global.rec_image_shape={m['shape']}",
    ]
    t0  = time.time()
    res = subprocess.run(cmd, cwd=str(PADDLEOCR_DIR),
                         capture_output=True, text=True, encoding="utf-8")
    elapsed = time.time() - t0
    raw     = res.stdout + res.stderr

    # Trả về list đã được reorder theo paths gốc
    mapped = _parse_paddle_log_mapped(raw, paths)
    return mapped, elapsed, raw


def run_trocr_inference(img_path: str):
    """Run TrOCR directly on GPU/CPU."""
    t0 = time.time()
    pv = _processor(Image.open(img_path).convert('RGB'),
                    return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        ids = _trocr.generate(
            pv, max_new_tokens=96, num_beams=4,
            early_stopping=True, no_repeat_ngram_size=3, repetition_penalty=1.3,
        )
    text = _tokenizer.decode(ids[0], skip_special_tokens=True)
    return [_normalize(text)], time.time() - t0, 'TrOCR OK'

print("✅ Inference functions ready")

✅ Inference functions ready


## 🔗 8 · Detection + OCR Pipeline

In [36]:
def detect_bubbles(img: np.ndarray):
    """Return (annotated_img, raw_crops, processed_strips, msg)."""
    raw     = _reader.readtext(img)
    results = filter_overlapping_boxes(raw, _cfg.iou_threshold)
    results.sort(key=lambda x: np.array(x[0]).min(axis=0)[1])

    annotated   = img.copy()
    raw_crops   = []   # original bbox crops  → TrOCR + gallery display
    proc_strips = []   # bubble_to_line output → PaddleOCR models

    for bbox, _, _ in results:
        pts = np.array(bbox, np.int32)
        x0, y0 = pts.min(axis=0).clip(0).astype(int)
        x1, y1 = pts.max(axis=0).clip(0).astype(int)
        crop = img[y0:y1, x0:x1]
        if crop.size > 0:
            raw_crops.append(crop)
            proc_strips.append(bubble_to_line(crop, _cfg))
        cv2.polylines(annotated, [pts.reshape(-1, 1, 2)], True, (0, 255, 0), 2)

    msg = f"Found {len(raw_crops)} bubble(s)"
    return annotated, raw_crops, proc_strips, msg


import shutil

def run_pipeline(img: np.ndarray, model_key: str):
    if img is None:
        return None, [], "⚠️ Please upload an image."
    if model_key not in MODEL_MAP:
        return img, [], f"❌ Unknown model: {model_key}"

    annotated, raw_crops, proc_strips, det_msg = detect_bubbles(img)
    if not raw_crops:
        return annotated, [], det_msg

    is_trocr = (model_key == "TrOCR")
    gallery, logs = [], [det_msg]

  # Xóa sạch thư mục trước khi chạy để tránh PaddleOCR đọc nhầm file rác
    if TEMP_DIR.exists():
        shutil.rmtree(TEMP_DIR)
    TEMP_DIR.mkdir(parents=True, exist_ok=True)

    # ── Save all bubble images to disk first ──────────────────────────
    paths = []
    for idx, (raw, proc) in enumerate(zip(raw_crops, proc_strips)):
        infer_input = raw if is_trocr else proc
        # 🚀 FIX 2: Ép tên file có 3 chữ số (000, 001... 010) để máy tính sort đúng
        path = str(TEMP_DIR / f"bubble_{idx:03d}.png") 
        cv2.imwrite(path, cv2.cvtColor(infer_input, cv2.COLOR_RGB2BGR))
        paths.append(path)

    # ── Single inference call for PaddleOCR (batch), per-image for TrOCR ──
    t0 = time.time()
    if is_trocr:
        all_preds = []
        for path in paths:
            preds, _, _ = run_trocr_inference(path)
            all_preds.append(preds)
        elapsed = time.time() - t0
        logs.append(f"TrOCR: {len(paths)} images in {elapsed:.2f}s")
    else:
        all_preds, elapsed, raw_log = run_paddle_inference(paths, model_key)    
        # ── Build gallery ─────────────────────────────────────────────────
        for idx, (raw, preds) in enumerate(zip(raw_crops, all_preds)):
            label = " | ".join(preds)
            logs.append(f"  [{idx}] {label}")
            gallery.append((raw, label))

    return annotated, gallery, "\n".join(logs)

print("✅ Pipeline ready")

✅ Pipeline ready


In [23]:
!ls /kaggle/working/temp_bubbles

bubble_000.png	bubble_005.png	bubble_010.png	bubble_015.png	bubble_020.png
bubble_001.png	bubble_006.png	bubble_011.png	bubble_016.png	bubble_021.png
bubble_002.png	bubble_007.png	bubble_012.png	bubble_017.png
bubble_003.png	bubble_008.png	bubble_013.png	bubble_018.png
bubble_004.png	bubble_009.png	bubble_014.png	bubble_019.png


## 🎨 9 · Gradio Interface

In [37]:
gr.close_all()

with gr.Blocks(title="Manga OCR") as demo:

    gr.Markdown("# 🈯 Manga OCR — Detection & Recognition")

    with gr.Row():
        with gr.Column(scale=1):
            inp_image = gr.Image(label="📥 Input Image", type="numpy")
            inp_model = gr.Dropdown(
                choices=list(MODEL_MAP.keys()),
                value="SVTR",
                label="🤖 Recognition Model",
            )
            btn_run = gr.Button("▶  Run Detection & OCR", variant="primary")

        with gr.Column(scale=1):
            out_annotated = gr.Image(label="🔍 Detected Bubbles")
            out_logs      = gr.Textbox(label="📋 Logs", lines=8)

    gr.Markdown("### 🫧 Bubble Crops & OCR Results")

    out_gallery = gr.Gallery(
        label="Bubbles", columns=2, height="auto",
        object_fit="contain", allow_preview=True,
    )

    # ✅ PHẢI đặt ở đây (bên trong Blocks)
    btn_run.click(
        fn=run_pipeline,
        inputs=[inp_image, inp_model],
        outputs=[out_annotated, out_gallery, out_logs],
        api_name="process_image"
    )

# launch bên ngoài
demo.launch(share=True)


Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://fdce80410e869715bb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

=== RAW PADDLE LOG (last 30 lines) ===
'[2026/04/25 09:31:29] ppocr INFO: \t result: おれ\t0.9993038177490234'
'[2026/04/25 09:31:29] ppocr INFO: infer_img: /kaggle/working/temp_bubbles/bubble_010.png'
'[2026/04/25 09:31:29] ppocr INFO: \t result: ははは\t0.9999251365661621'
'[2026/04/25 09:31:29] ppocr INFO: infer_img: /kaggle/working/temp_bubbles/bubble_011.png'
'[2026/04/25 09:31:29] ppocr INFO: \t result: 肉は\t0.9999886155128479'
'[2026/04/25 09:31:29] ppocr INFO: infer_img: /kaggle/working/temp_bubbles/bubble_012.png'
'[2026/04/25 09:31:29] ppocr INFO: \t result: やはははは\t0.9328889846801758'
'[2026/04/25 09:31:29] ppocr INFO: infer_img: /kaggle/working/temp_bubbles/bubble_013.png'
'[2026/04/25 09:31:29] ppocr INFO: \t result: ははは\t0.9994551539421082'
'[2026/04/25 09:31:29] ppocr INFO: infer_img: /kaggle/working/temp_bubbles/bubble_014.png'
'[2026/04/25 09:31:29] ppocr INFO: \t result: おれはケガだってせん浩くない連れてくよ\t0.8605796694755554'
'[2026/04/25 09:31:29] ppocr INFO: infer_img: /kaggle/working/te

## 🧪 10 · Quick Test



Runs all three models on a synthetic bubble image to verify the pipeline

end-to-end without needing a real manga page.

In [ ]:
dummy = np.zeros((64, 256, 3), dtype=np.uint8)
cv2.putText(dummy, "TEST OCR", (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
test_path = str(TEMP_DIR / 'test_bubble.png')
cv2.imwrite(test_path, dummy)
display(Image.open(test_path))

for key in ("SVTR", "CRNN", "TrOCR"):
    try:
        fn = (run_trocr_inference if key == 'TrOCR'
              else lambda p, k=key: run_paddle_inference(p, k))
        preds, elapsed, _ = fn(test_path)
        print(f'  {key:<6} -> {preds}  ({elapsed:.2f}s)')
    except Exception as err:
        print(f'  {key:<6} -> ERROR: {err}')